# SD 1.5 → Qualcomm NPU (Local Dream / Ruya) — RESMİ HAT

Bu defter, Local Dream'in **kendi dönüştürme scriptlerini** (`npuconvertv2`)
QNN SDK **2.28** ile koşar. Rehber: `ld-guide.chino.icu/conversion/sd15`

**Neden bu hat:** kendi yazdığımız hat cihazda yüklenmeyen paketler üretiyordu.
Sebepleri resmi scriptlerde görüldü:

| bizim (eski) | resmi |
|---|---|
| `qairt-converter` → DLC | `qnn-onnx-converter` → `model.cpp` → `.so` |
| `--act_bitwidth 8` + io16 hilesi | **`--act_bitwidth 16`** |
| per-channel kapalı | `--use_per_channel_quantization` |
| stok diffusers | `redefined_modules/` (MHA→SHA, Linear→Conv) |
| VTCM ayarsız | `"vtcm_mb": 2` |

**Çalışma sırası:** 1 → 2 → 3 → 4 → 5

> **GPU notu:** GPU yalnızca kalibrasyon verisi üretimini (`prepare_data.py`)
> hızlandırır — ~35 dk yerine ~3 dk. Kuantizasyon, model-lib ve context-binary
> adımları **tamamen CPU**'dur ve GPU'dan etkilenmez. Yine de yüksek RAM
> gerektiği için (rehber: ~20 GB) yüksek bellekli çalışma zamanı şart.


## 1) Ayarlar

In [ ]:
#@title Ayarlar { display-mode: "form" }
#@markdown Model dosyasının **doğrudan indirme** bağlantısı (.safetensors)
SAFETENSORS_URL = ""  #@param {type:"string"}
MODEL_NAME = "MyModel"  #@param {type:"string"}
#@markdown Çip katmanı — `min` = Hexagon V68+ (Snapdragon 7 Gen 1 dahil)
SOC = "min"  #@param ["min", "8gen1", "8gen2"]
#@markdown `clip_skip`: modelin eğitildiği değer. Anime çoğunlukla 2.
CLIP_SKIP = 2  #@param [1, 2] {type:"raw"}
#@markdown Foto-gerçekçi model ise işaretleyin (kalibrasyon promptlarını değiştirir)
REALISTIC = True  #@param {type:"boolean"}
#@markdown Kuantizasyon örnek sayısı. Resmi tarif 400 (saatler).
#@markdown `24` = boru hattını doğrula, `150` = iyi denge, `0` = tam (400)
CALIB_LIMIT = 150  #@param {type:"integer"}
#@markdown GPU varsa CUDA torch kur (prepare_data ~10x hızlanır)
CUDA_TORCH = True  #@param {type:"boolean"}

import os
os.environ.update(
    MODEL_NAME=MODEL_NAME, SOC=SOC,
    CLIP_SKIP=str(CLIP_SKIP),
    REALISTIC="1" if REALISTIC else "0",
    CALIB_LIMIT=str(CALIB_LIMIT),
    CUDA_TORCH="1" if CUDA_TORCH else "0",
    SAFETENSORS_URL=SAFETENSORS_URL,
)
print(f"{MODEL_NAME} | soc={SOC} clip_skip={CLIP_SKIP} realistic={REALISTIC}")
print(f"calib_limit={CALIB_LIMIT} cuda_torch={CUDA_TORCH}")
!nvidia-smi -L || echo "GPU yok — prepare_data yavas olacak"


## 2) Depo + araçlar

Depoyu klonlar/günceller ve `uv`'yi kurar. Çalışma zamanı sıfırlansa bile
tekrar çalıştırmak yeterli.

In [ ]:
%cd /content
REPO = "https://github.com/matrixportalx/Sd-1.5-Converting-to-Qualcomm-QNN-Model"
BRANCH = "claude/qnn-model-conversion-snapdragon7-rsk8og"
import os, subprocess
if not os.path.isdir("/content/sd-qnn/.git"):
    !git clone -b {BRANCH} {REPO} /content/sd-qnn
else:
    !cd /content/sd-qnn && git fetch origin {BRANCH} && git reset --hard origin/{BRANCH}
%cd /content/sd-qnn
!pip install -q uv
!git log --oneline -1


## 3) QNN SDK 2.28

~2 GB. **Sürüm önemli** — rehber 2.28 şart koşuyor. İndirme koparsa bu hücreyi
tekrar çalıştırın, kaldığı yerden devam eder.

In [ ]:
%cd /content/sd-qnn
import os
URL = ("https://apigwx-aws.qualcomm.com/qsc/public/v1/api/download/software/"
       "qualcomm_neural_processing_sdk/v2.28.0.241029.zip")
out = !python3 scripts/setup_qnn_sdk.py --dest /content/qairt --asset-url "{URL}"
print("\n".join(out[-25:]))
root = [l.split("=",1)[1] for l in out if l.startswith("QNN_SDK_ROOT=")]
assert root, "QNN_SDK_ROOT bulunamadi — yukaridaki ciktiya bakin"
os.environ["QNN_SDK_ROOT"] = root[-1].strip()
print("\nQNN_SDK_ROOT =", os.environ["QNN_SDK_ROOT"])


## 4) Modeli indir

Resmi hat `.safetensors` dosyasını **doğrudan** kullanır.

In [ ]:
%cd /content/sd-qnn
import os
os.makedirs("work", exist_ok=True)
CKPT = "work/input.safetensors"
url = os.environ.get("SAFETENSORS_URL", "").strip()
if os.path.exists(CKPT) and os.path.getsize(CKPT) > 100_000_000:
    print(f"[ATLANDI] {CKPT} zaten var ({os.path.getsize(CKPT)/1e9:.2f} GB)")
elif not url:
    raise SystemExit("SAFETENSORS_URL bos — 1. hucredeki ayarlari doldurun")
else:
    !wget -c -O {CKPT} "{url}"
    print(f"[+] {os.path.getsize(CKPT)/1e9:.2f} GB")


## 5) Dönüştür

Aşamalar: `uv` ortamı → `prepare_data` → `gen_quant_data` → `export_onnx` →
`qnn-onnx-converter` → `qnn-model-lib-generator` → `qnn-context-binary-generator`

`data.pkl` ve `unet/model.onnx` önbelleğe alınır: `CALIB_LIMIT` değiştirip
tekrar çalıştırırsanız veri üretimi **atlanır**, sadece kuantizasyon yenilenir.

In [ ]:
%cd /content/sd-qnn
import os
assert os.environ.get("QNN_SDK_ROOT"), "Once 3. hucreyi calistirin"
name = os.environ["MODEL_NAME"]
!bash scripts/06_official_pipeline.sh work/input.safetensors "{name}" "work/{name}" "$SOC"


## 6) Paketi indir

`BITTI` satırını gördükten sonra çalıştırın. **Hemen indirin** — çalışma zamanı
kapanırsa dosya kaybolur.

In [ ]:
import glob, os
from google.colab import files
zips = sorted(glob.glob("/content/sd-qnn/dist/*.zip"), key=os.path.getmtime)
assert zips, "dist/ bos — donusum tamamlanmadi"
z = zips[-1]
print(f"{z}  ({os.path.getsize(z)/1e6:.0f} MB)")
!unzip -l "{z}"
files.download(z)
